In [4]:
from pathlib import Path
import pandas as pd

data_raw_dir = Path("../data/raw")
files = {
    "stunting":       data_raw_dir / "jme_expand_database_stunting_2025.xlsx",
    "wasting":        data_raw_dir / "jme_expand_database_wasting_2025.xlsx",
    "severe_wasting": data_raw_dir / "jme_expand_database_severe_wasting_2025.xlsx",
}

DEMO_GROUPS = [
    "National", "Male", "Female",
    "Urban", "Rural",
    "Wealth Quintile 1", "Wealth Quintile 2", "Wealth Quintile 3",
    "Wealth Quintile 4", "Wealth Quintile 5",
    "Wealth Quintile 1 Urban", "Wealth Quintile 1 Rural",
    "Wealth Quintile 2 Urban", "Wealth Quintile 2 Rural",
    "Wealth Quintile 3 Urban", "Wealth Quintile 3 Rural",
    "Wealth Quintile 4 Urban", "Wealth Quintile 4 Rural",
    "Wealth Quintile 5 Urban", "Wealth Quintile 5 Rural",
]

WEALTH_MOD  = {"Wealth Quintile 1": 30, "Wealth Quintile 2": 20,
               "Wealth Quintile 3": 10, "Wealth Quintile 4": 5, "Wealth Quintile 5": 0}
RESID_MOD   = {"Rural": 15, "Urban": -5}
BASE_COSTS  = {"stunting": 20, "wasting": 25, "severe_wasting": 30}


# ── helpers ──────────────────────────────────────────────────────────────────

def load_metric(file_path, metric):
    df = pd.read_excel(file_path, sheet_name="Trend", skiprows=6, header=[0, 1])
    df = df.iloc[1:].reset_index(drop=True)

    iso_col     = next(c for c in df.columns if c[0] == "ISO")
    country_col = next(c for c in df.columns if c[0] == "Countries and areas")
    ids = df[[iso_col, country_col]].copy()
    ids.columns = ["ISO3", "Country"]

    available = set(df.columns.get_level_values(0))
    rows = []
    for demo in DEMO_GROUPS:
        if demo not in available:
            continue
        col = (demo, "Point Estimate")
        if col not in df.columns:
            continue
        values = df[col].values
        for i, val in enumerate(values):
            try:
                rows.append({"ISO3": ids.iloc[i]["ISO3"], "Country": ids.iloc[i]["Country"],
                             "Demographic_group": demo, f"Point_estimate_{metric}": float(val)})
            except (ValueError, TypeError):
                pass

    return pd.DataFrame(rows)


def mece_population(demo, total_pop):
    rural_share = 0.40 if "Rural" in demo else 0.60
    return total_pop * rural_share / 5


def intervention_cost(demo, metric):
    w = next((v for k, v in WEALTH_MOD.items() if k in demo), 0)
    r = next((v for k, v in RESID_MOD.items() if k in demo), 0)
    return BASE_COSTS[metric] + w + r


# ── pipeline ─────────────────────────────────────────────────────────────────

# 1. Extract disaggregated data for each metric and merge
group_cols = ["ISO3", "Country", "Demographic_group"]

df_stunt  = load_metric(files["stunting"],       "stunting")
df_wast   = load_metric(files["wasting"],        "wasting")
df_severe = load_metric(files["severe_wasting"], "severe_wasting")

df = (df_stunt
      .merge(df_wast,   on=group_cols)
      .merge(df_severe, on=group_cols))

value_cols = ["Point_estimate_stunting", "Point_estimate_wasting", "Point_estimate_severe_wasting"]
df = df.groupby(group_cols)[value_cols].mean().reset_index()

# 2. Add UN under-5 population (2023)
pop_file = Path("../data/raw/WPP2024_POP_F02_1_POPULATION_5-YEAR_AGE_GROUPS_BOTH_SEXES.xlsx")
pop = pd.read_excel(pop_file, skiprows=16)
pop = (pop[(pop["Type"] == "Country/Area") & (pop["Year"] == 2023)]
       [["ISO3 Alpha-code", "0-4"]]
       .rename(columns={"ISO3 Alpha-code": "ISO3", "0-4": "Population_u5"}))
pop["Population_u5"] = pd.to_numeric(pop["Population_u5"], errors="coerce")
pop["Population_u5"] = (pop["Population_u5"] * 1000).round().astype("Int64")

df = df.merge(pop, on="ISO3", how="left")

# 3. Keep only MECE rows: Wealth Quintile X + Urban/Rural
is_mece = df["Demographic_group"].apply(
    lambda g: any(f"Wealth Quintile {i}" in g for i in range(1, 6))
              and ("Urban" in g or "Rural" in g)
)
df = df[is_mece].reset_index(drop=True)

# 4. MECE population split: (pop × 0.4 or 0.6) / 5
df["Population_u5_adjusted"] = (
    df.apply(lambda r: mece_population(r["Demographic_group"], r["Population_u5"]), axis=1)
    .round().astype("Int64")
)

# 5. Absolute burden counts
for metric in ["stunting", "wasting", "severe_wasting"]:
    df[f"Count_{metric}"] = (
        (df[f"Point_estimate_{metric}"] / 100 * df["Population_u5_adjusted"])
        .round().astype("Int64")
    )

# 6. Intervention costs
for metric in ["stunting", "wasting", "severe_wasting"]:
    df[f"Cost_{metric}"] = df["Demographic_group"].apply(
        lambda g, m=metric: intervention_cost(g, m)
    )

# ── save ─────────────────────────────────────────────────────────────────────

out = Path("../data/processed/master_df_mece_compliant.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)

print(f"Saved {len(df)} rows × {df.shape[1]} columns → {out}")
print(df.head(3).to_string())

Saved 1590 rows × 14 columns → ..\data\processed\master_df_mece_compliant.csv
  ISO3      Country        Demographic_group  Point_estimate_stunting  Point_estimate_wasting  Point_estimate_severe_wasting  Population_u5  Population_u5_adjusted  Count_stunting  Count_wasting  Count_severe_wasting  Cost_stunting  Cost_wasting  Cost_severe_wasting
0  AFG  Afghanistan  Wealth Quintile 1 Rural                     49.3                     5.4                            1.3        6656181                  532494          262520          28755                  6922             65            70                   75
1  AFG  Afghanistan  Wealth Quintile 1 Urban                     68.9                     4.4                            2.7        6656181                  798742          550333          35145                 21566             45            50                   55
2  AFG  Afghanistan  Wealth Quintile 2 Rural                     41.2                     5.5                            